In [ ]:
import pandas as pd
import torch

# Per riproducibilità
torch.manual_seed(8347247)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("../data/all_ships.csv")

In [3]:
from sklearn.model_selection import train_test_split

X = df[["sex", "age", "age_missing", "class", "crew"]]
Y = df["survived"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
Y_tensor = torch.tensor(Y.values, dtype=torch.long)

# per riproducibilità si usa random_state fissato
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42, stratify=Y_tensor)

mean = X_train.mean(0)
std = X_train.std(0)

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

In [4]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_norm, Y_train)
test_ds = TensorDataset(X_test_norm, Y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [ ]:
from torch import nn

class SoftMaxRegressor(nn.Module):
    """
    Modello softmax lineare:
    - livello fully connected X -> classi
    - senza attivazione finale (CrossEntropyLoss la applica internamente)
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x) # ritorna logits (punteggi non normalizzati)

In [6]:
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score

def train_model(model, train_loader, test_loader, lr=0.05, epochs=300):
    writer = SummaryWriter(f'../results/{model._get_name()}')
    model = model.to(device)

    weights = torch.tensor([1.0, 2.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.001)

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        y_true = []
        y_pred = []
        
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)
            loss = criterion(output, Y_batch)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() * X_batch.size(0)

            # l'output è array di logits quindi si prende il punteggio più alto che indica la classe più probabile
            preds = output.argmax(dim=1)
            y_true.extend(Y_batch.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)

        model.eval()

        test_loss = 0.0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)

                loss = criterion(output, Y_batch)
                test_loss += loss.item() * X_batch.size(0)
                
                preds = output.argmax(dim=1)
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        test_acc = accuracy_score(y_true, y_pred)

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('accuracy/train', train_acc, epoch)
        writer.add_scalar('loss/test', test_loss, epoch)
        writer.add_scalar('accuracy/test', test_acc, epoch)

        if epoch % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")
    writer.close()

    return model, train_loss, train_acc, test_loss, test_acc

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(model):
    model.eval()

    with torch.no_grad():
        output = model(X_test_norm.to(device))
        probs = output.argmax(dim=1)

    y_true = Y_test.cpu().numpy()
    y_pred = probs.cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return accuracy, precision, recall, f1

In [8]:
model = SoftMaxRegressor(in_features=5, out_features=2)
model, train_loss, train_acc, test_loss, test_acc = train_model(model, train_loader, test_loader)
accuracy, precision, recall, f1 = evaluate_model(model)

Epoch 1/300 | train_loss 0.6892 | train_acc 0.5768 | test_loss 0.6744 | test_acc 0.5657
Epoch 51/300 | train_loss 0.6836 | train_acc 0.5804 | test_loss 0.6652 | test_acc 0.5969
Epoch 101/300 | train_loss 0.6857 | train_acc 0.5713 | test_loss 0.6774 | test_acc 0.5163
Epoch 151/300 | train_loss 0.6862 | train_acc 0.5830 | test_loss 0.6974 | test_acc 0.5475
Epoch 201/300 | train_loss 0.6861 | train_acc 0.5654 | test_loss 0.6780 | test_acc 0.6294
Epoch 251/300 | train_loss 0.6864 | train_acc 0.5778 | test_loss 0.6837 | test_acc 0.5904
Epoch 300/300 | train_loss 0.6842 | train_acc 0.5742 | test_loss 0.6923 | test_acc 0.5969


In [9]:
print("Model metrics\n")

print(f"train_loss: {train_loss:.4f}")
print(f"train_acc: {train_acc:.4f}")
print(f"test_loss: {test_loss:.4f}")
print(f"test_acc: {test_acc:.4f}\n")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Model metrics

train_loss: 0.6842
train_acc: 0.5742
test_loss: 0.6923
test_acc: 0.5969

Accuracy: 0.5969
Precision: 0.4148
Recall: 0.5840
F1 Score: 0.4850
